# DeepSeek-OCR-2: Visual Causal Flow

**Paper:** [arXiv:2601.20552](https://arxiv.org/abs/2601.20552)

**Requirements:** GPU runtime (T4 or better), ~15GB GPU memory

## 1. Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies (specific versions for DeepSeek-OCR-2)
!pip install -q transformers==4.45.0 accelerate>=0.34.0 pillow addict attrdict einops timm
!pip install -q flash-attn --no-build-isolation

print("✓ Dependencies installed. Restart runtime if you see import errors.")

## 2. Download Sample Images from GitHub

In [ ]:
import os
import urllib.request

# Create directories
os.makedirs("test-images", exist_ok=True)
os.makedirs("results/sidebyside", exist_ok=True)
os.makedirs("output", exist_ok=True)

# Download sample images from GitHub
GITHUB_RAW = "https://raw.githubusercontent.com/maycuatroi1/ocr-comparison/master/test-images"

sample_images = [
    "crazy-hand-writing.png",
    "wild.png",
    "Document.jpg"
]

print("Downloading sample images from GitHub...")
for img_name in sample_images:
    url = f"{GITHUB_RAW}/{img_name}"
    dest = f"test-images/{img_name}"
    try:
        urllib.request.urlretrieve(url, dest)
        print(f"  ✓ {img_name}")
    except Exception as e:
        print(f"  ✗ {img_name}: {e}")

print(f"\nImages in test-images/: {os.listdir('test-images')}")

## 3. Load DeepSeek-OCR-2 Model

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import os

os.environ["CUDA_VISIBLE_DEVICES"] = '0'

MODEL_NAME = "deepseek-ai/DeepSeek-OCR-2"

print(f"Loading model: {MODEL_NAME}")
print("This may take a few minutes...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    _attn_implementation='flash_attention_2',
    use_safetensors=True
)
model = model.eval().cuda().to(torch.bfloat16)

print("\n✓ Model loaded successfully!")

## 4. Define OCR and Visualization Functions

In [ ]:
import time
import json
import textwrap
from datetime import datetime
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

def perform_ocr(image_path: str) -> tuple:
    """Perform OCR and return (text, processing_time)"""
    start = time.time()
    try:
        # Use the correct inference method
        prompt = "<image>\nTrích xuất TẤT CẢ văn bản từ hình ảnh này, bao gồm cả chữ viết tay."
        
        result = model.infer(
            tokenizer,
            prompt=prompt,
            image_file=image_path,
            output_path="output",
            base_size=1024,
            image_size=768,
            crop_mode=True,
            save_results=False
        )
        
        elapsed = time.time() - start
        # Result might be a string or dict depending on version
        if isinstance(result, dict):
            text = result.get('text', str(result))
        else:
            text = str(result)
        return text, elapsed
    except Exception as e:
        return f"Error: {str(e)}", time.time() - start


def get_font(size=14):
    """Get font for drawing"""
    try:
        return ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf", size)
    except:
        return ImageFont.load_default()


def create_sidebyside(image_path: Path, ocr_text: str, processing_time: float, output_path: Path):
    """Create side-by-side visualization: Original | OCR Results"""
    
    # Load and resize original image
    original = Image.open(image_path).convert("RGB")
    target_height = 800
    ratio = target_height / original.height
    new_width = int(original.width * ratio)
    original_resized = original.resize((new_width, target_height), Image.Resampling.LANCZOS)

    # Create text panel
    text_panel_width = 600
    text_panel = Image.new('RGB', (text_panel_width, target_height), (30, 30, 30))
    draw = ImageDraw.Draw(text_panel)

    # Fonts
    title_font = get_font(20)
    text_font = get_font(14)
    small_font = get_font(12)

    # Draw title
    draw.text((15, 15), "OCR Results (DeepSeek-OCR-2)", fill="#4ECDC4", font=title_font)
    draw.line([(15, 45), (text_panel_width - 15, 45)], fill="#4ECDC4", width=2)

    # Wrap and draw OCR text
    y_offset = 60
    line_height = 20
    max_chars = 55

    lines = ocr_text.split('\n')
    line_num = 1

    for line in lines:
        if not line.strip():
            y_offset += line_height // 2
            continue

        wrapped = textwrap.wrap(line, width=max_chars) or ['']

        for i, wrapped_line in enumerate(wrapped):
            if y_offset > target_height - 40:
                draw.text((15, y_offset), "... [truncated]", fill="#888888", font=text_font)
                break

            if i == 0:
                draw.text((15, y_offset), f"{line_num:2d}.", fill="#888888", font=text_font)
                line_num += 1

            draw.text((50, y_offset), wrapped_line, fill="white", font=text_font)
            y_offset += line_height

        if y_offset > target_height - 40:
            break

    # Add stats at bottom
    stats_y = target_height - 30
    draw.line([(15, stats_y - 10), (text_panel_width - 15, stats_y - 10)], fill="#444444", width=1)
    num_lines = len([l for l in lines if l.strip()])
    num_chars = len(ocr_text)
    draw.text((15, stats_y), f"⏱ {processing_time:.2f}s | Lines: {num_lines} | Chars: {num_chars} | DeepSeek-OCR-2",
              fill="#4ECDC4", font=small_font)

    # Combine images
    divider_width = 3
    total_width = new_width + divider_width + text_panel_width
    combined = Image.new('RGB', (total_width, target_height), (60, 60, 60))

    combined.paste(original_resized, (0, 0))
    
    # Draw divider
    divider_draw = ImageDraw.Draw(combined)
    divider_draw.rectangle([new_width, 0, new_width + divider_width, target_height], fill="#4ECDC4")

    combined.paste(text_panel, (new_width + divider_width, 0))

    # Save
    output_path.parent.mkdir(parents=True, exist_ok=True)
    combined.save(output_path, quality=95)

    return combined


print("✓ OCR and visualization functions defined")

## 5. Run OCR and Generate Side-by-Side Visualizations

In [ ]:
IMAGE_DIR = Path("test-images")
OUTPUT_DIR = Path("results/sidebyside")

image_extensions = {".png", ".jpg", ".jpeg", ".webp"}
image_files = [f for f in IMAGE_DIR.iterdir() if f.suffix.lower() in image_extensions]

print(f"Processing {len(image_files)} images...")
print("=" * 60)

all_results = {}

for image_path in sorted(image_files):
    print(f"\n📷 {image_path.name}")
    
    # Get absolute path
    abs_path = str(image_path.absolute())
    
    # Perform OCR
    print("  Extracting text...")
    ocr_text, processing_time = perform_ocr(abs_path)
    print(f"  ⏱ Processing time: {processing_time:.2f}s")
    
    # Preview
    preview = ocr_text[:100].replace('\n', ' ')
    print(f"  Preview: {preview}...")
    
    # Create side-by-side visualization
    output_path = OUTPUT_DIR / f"{image_path.stem}_sidebyside.png"
    vis_img = create_sidebyside(image_path, ocr_text, processing_time, output_path)
    print(f"  ✓ Saved: {output_path.name}")
    
    # Display in notebook
    display(vis_img.resize((vis_img.width // 2, vis_img.height // 2)))
    
    # Save raw text
    txt_path = OUTPUT_DIR / f"{image_path.stem}_ocr.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(ocr_text)
    
    # Store results
    all_results[image_path.name] = {
        "text": ocr_text,
        "processing_time": processing_time,
        "timestamp": datetime.now().isoformat()
    }

# Save JSON results
with open(OUTPUT_DIR / "deepseek_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 60)
print("✓ All images processed!")
print(f"Results saved in: {OUTPUT_DIR}")

## 6. Download Results

In [ ]:
from google.colab import files

# Zip all results
!zip -r deepseek_ocr_results.zip results/

print("\nDownloading results...")
files.download("deepseek_ocr_results.zip")

## 7. (Optional) Upload Your Own Images

In [ ]:
from google.colab import files

print("Upload your images:")
uploaded = files.upload()

for filename in uploaded.keys():
    # Save to test-images
    dest = f"test-images/{filename}"
    if os.path.exists(filename):
        os.rename(filename, dest)
    
    print(f"\n📷 Processing: {filename}")
    
    # Run OCR with absolute path
    abs_path = os.path.abspath(dest)
    ocr_text, processing_time = perform_ocr(abs_path)
    print(f"  ⏱ Time: {processing_time:.2f}s")
    
    # Create visualization
    output_path = OUTPUT_DIR / f"{Path(filename).stem}_sidebyside.png"
    vis_img = create_sidebyside(Path(dest), ocr_text, processing_time, output_path)
    
    # Display
    display(vis_img.resize((vis_img.width // 2, vis_img.height // 2)))
    
    print(f"\nExtracted text:\n{'='*40}")
    print(ocr_text)